In [ ]:
import sys, glob, os, json, shutil, subprocess
REPO = glob.glob("/kaggle/input/**/arcagi-arc-agi-1", recursive=True)[0]
sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO
MODEL = "/kaggle/working/model"
sb = glob.glob("/kaggle/input/**/stage_b_examples.json", recursive=True)
assert sb, "stage_b_examples.json not found — attach Commit A's notebook Output as input"
STAGE_B = sb[0]
print("REPO   :", REPO)
print("STAGE_B:", STAGE_B, "->", len(json.load(open(STAGE_B))), "examples")

In [ ]:
subprocess.run([sys.executable, "scripts/meta_train.py","--stage","b",
                "--build","build","--examples",STAGE_B,
                "--out",MODEL,"--device","cuda","--steps","1500"],
               cwd=REPO, env=os.environ, check=True)

In [ ]:
os.makedirs(MODEL, exist_ok=True)
for f in ["backbone_a.pt","prior_a.pt"]:
    shutil.copy(f"{REPO}/model/{f}", f"{MODEL}/{f}")
print(sorted(os.listdir(MODEL)))   # want all 5: backbone_a/b, prior_a/b, vocab.json

In [ ]:
import random, statistics
from pathlib import Path
from arcagi.checkpoint import load_stages
from arcagi.config import Config
from arcagi.stage_c import solve
from arcagi.sandbox import PersistentSandbox
from arcagi.evaluate import evaluate_submission

cfg = Config()
stages = load_stages(Path(MODEL), cfg.model, cfg.prefix)
assert stages, "load_stages None — check all 5 files are in MODEL"

tasks     = json.loads(Path(f"{REPO}/build/eval_tasks.json").read_text())
solutions = json.loads(Path(f"{REPO}/build/eval_solutions.json").read_text())
LIMIT = 20
task_ids = list(tasks)[:LIMIT]
submission, rng = {}, random.Random(0)
with PersistentSandbox(cfg.sandbox) as sandbox:
    for i, tid in enumerate(task_ids, 1):
        t = tasks[tid]
        demo = [(p["input"], p["output"]) for p in t["train"]]
        entries = []
        for test in t["test"]:
            attempts, _ = solve(stages, demo, test["input"], sandbox, cfg.stage_c,
                                k=cfg.stage_c.n_submission_attempts, rng=rng)
            entries.append({f"attempt_{j+1}": g for j, g in enumerate(attempts)})
        submission[tid] = entries
        print(f"[{i}/{len(task_ids)}] {tid}", flush=True)

report = evaluate_submission(submission, {k: solutions[k] for k in task_ids})
items = report.items; n = len(items)
print(f"\npass@1/2/3     : {report.pass_at(1):.3f}/{report.pass_at(2):.3f}/{report.pass_at(3):.3f}")
print(f"mean_best_cell : {report.mean_best_cell_accuracy:.3f}   <- leading indicator")
print(f"execution_rate : {sum(t.any_executed for t in items)/n:.3f}")
print(f"pools nonempty : {sum(t.any_executed for t in items)}/{n}   <- was 0 before the fix")